In [ ]:
!pip install pymupdf pandas openpyxl tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/CSAI415_D1"
PDF_DIR = os.path.join(BASE_DIR, "pdfs")
EXCEL_PATH = os.path.join(BASE_DIR, "Dataset Special Topics.xlsx")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("PDF folder:", PDF_DIR)
print("Excel file:", EXCEL_PATH)
print("Output folder:", OUTPUT_DIR)

PDF folder: /content/drive/MyDrive/CSAI415_D1/pdfs
Excel file: /content/drive/MyDrive/CSAI415_D1/Dataset Special Topics.xlsx
Output folder: /content/drive/MyDrive/CSAI415_D1/outputs


In [ ]:
import pandas as pd

df = pd.read_excel(EXCEL_PATH)
df.head()

,paper_id,paper_title_to_search,topic,subtopic,priority,recommended_source,recommended_filename,downloaded? Y/N,final_pdf_path,notes
0,paper_001,Retrieval-Augmented Generation for Knowledge-I...,RAG,Foundational RAG,High,arXiv / NeurIPS,retrieval_augmented_generation_for_knowledge_i...,y,NaN,Search exact title first; download PDF if rele...
1,paper_002,Retrieval-Augmented Generation for Large Langu...,RAG,Survey,High,arXiv,retrieval_augmented_generation_for_large_langu...,y,NaN,Search exact title first; download PDF if rele...
2,paper_003,REALM: Retrieval-Augmented Language Model Pre-...,RAG,Retrieval pretraining,High,arXiv / ICML,realm_retrieval_augmented_language_model_pre_t...,y,NaN,Search exact title first; download PDF if rele...
3,paper_004,Retrieval-Augmented Language Model Pre-Training,RAG,Retrieval pretraining,High,arXiv,retrieval_augmented_language_model_pre_trainin...,y,NaN,Search exact title first; download PDF if rele...
4,paper_005,Retrieval-Augmented Multimodal Language Modeling,RAG,Multimodal RAG,Medium,arXiv,retrieval_augmented_multimodal_language_modeli...,y,NaN,Search exact title first; download PDF if rele...


In [ ]:
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")
df.head()

,paper_id,paper_title_to_search,topic,subtopic,priority,recommended_source,recommended_filename,downloaded?_y/n,final_pdf_path,notes
0,paper_001,Retrieval-Augmented Generation for Knowledge-I...,RAG,Foundational RAG,High,arXiv / NeurIPS,retrieval_augmented_generation_for_knowledge_i...,y,NaN,Search exact title first; download PDF if rele...
1,paper_002,Retrieval-Augmented Generation for Large Langu...,RAG,Survey,High,arXiv,retrieval_augmented_generation_for_large_langu...,y,NaN,Search exact title first; download PDF if rele...
2,paper_003,REALM: Retrieval-Augmented Language Model Pre-...,RAG,Retrieval pretraining,High,arXiv / ICML,realm_retrieval_augmented_language_model_pre_t...,y,NaN,Search exact title first; download PDF if rele...
3,paper_004,Retrieval-Augmented Language Model Pre-Training,RAG,Retrieval pretraining,High,arXiv,retrieval_augmented_language_model_pre_trainin...,y,NaN,Search exact title first; download PDF if rele...
4,paper_005,Retrieval-Augmented Multimodal Language Modeling,RAG,Multimodal RAG,Medium,arXiv,retrieval_augmented_multimodal_language_modeli...,y,NaN,Search exact title first; download PDF if rele...


In [ ]:
if "paper_id" not in df.columns:
    df["paper_id"] = range(1, len(df) + 1)

df.head()

,paper_id,paper_title_to_search,topic,subtopic,priority,recommended_source,recommended_filename,downloaded?_y/n,final_pdf_path,notes
0,paper_001,Retrieval-Augmented Generation for Knowledge-I...,RAG,Foundational RAG,High,arXiv / NeurIPS,retrieval_augmented_generation_for_knowledge_i...,y,NaN,Search exact title first; download PDF if rele...
1,paper_002,Retrieval-Augmented Generation for Large Langu...,RAG,Survey,High,arXiv,retrieval_augmented_generation_for_large_langu...,y,NaN,Search exact title first; download PDF if rele...
2,paper_003,REALM: Retrieval-Augmented Language Model Pre-...,RAG,Retrieval pretraining,High,arXiv / ICML,realm_retrieval_augmented_language_model_pre_t...,y,NaN,Search exact title first; download PDF if rele...
3,paper_004,Retrieval-Augmented Language Model Pre-Training,RAG,Retrieval pretraining,High,arXiv,retrieval_augmented_language_model_pre_trainin...,y,NaN,Search exact title first; download PDF if rele...
4,paper_005,Retrieval-Augmented Multimodal Language Modeling,RAG,Multimodal RAG,Medium,arXiv,retrieval_augmented_multimodal_language_modeli...,y,NaN,Search exact title first; download PDF if rele...


In [ ]:
pdf_files = [f for f in os.listdir(PDF_DIR) if f.lower().endswith(".pdf")]

print("Number of PDFs:", len(pdf_files))
print(pdf_files[:10])

Number of PDFs: 99
['retrieval_augmented_generation_for_knowledge_intensive_nlp_tasks.pdf', 'retrieval_augmented_generation_for_large_language_models_a_survey.pdf', 'realm_retrieval_augmented_language_model_pre_training.pdf', 'retrieval_augmented_language_model_pre_training.pdf', 'retrieval_augmented_multimodal_language_modeling.pdf', 'atlas_few_shot_learning_with_retrieval_augmented_language_models.pdf', 'retro_improving_language_models_by_retrieving_from_trillions_of_tokens.pdf', 'fusion_in_decoder_leveraging_passage_retrieval_with_generative_models_.pdf', 'self_rag_learning_to_retrieve_generate_and_critique_through_self_refle.pdf', 'corrective_retrieval_augmented_generation.pdf']


In [ ]:
import fitz
from tqdm import tqdm

def extract_pdf_text(pdf_path):
    pages = []

    try:
        doc = fitz.open(pdf_path)

        for page_num, page in enumerate(doc, start=1):
            text = page.get_text()

            if text.strip():
                pages.append({
                    "page": page_num,
                    "text": text
                })

        doc.close()

    except Exception as e:
        print("Error reading:", pdf_path)
        print(e)

    return pages

In [ ]:
documents = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    paper_id = row["paper_id"]
    title = str(row.get("title", "")).strip()

    matched_pdf = None

    for pdf in pdf_files:
        clean_title = title[:40].lower().replace(":", "").replace("/", "").replace("-", " ")
        clean_pdf = pdf.lower().replace(":", "").replace("/", "").replace("-", " ")

        if clean_title in clean_pdf:
            matched_pdf = pdf
            break

    if matched_pdf is None:
        continue

    pdf_path = os.path.join(PDF_DIR, matched_pdf)
    pages = extract_pdf_text(pdf_path)

    documents.append({
        "paper_id": paper_id,
        "title": title,
        "pdf_file": matched_pdf,
        "pages": pages
    })

print("Matched documents:", len(documents))

100%|██████████| 100/100 [00:13<00:00,  7.25it/s]

Matched documents: 100


In [ ]:
documents = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    if i >= len(pdf_files):
        break

    paper_id = row["paper_id"]
    title = str(row.get("title", "")).strip()
    matched_pdf = pdf_files[i]

    pdf_path = os.path.join(PDF_DIR, matched_pdf)
    pages = extract_pdf_text(pdf_path)

    documents.append({
        "paper_id": paper_id,
        "title": title,
        "pdf_file": matched_pdf,
        "pages": pages
    })

print("Documents created:", len(documents))

 99%|█████████▉| 99/100 [04:28<00:02,  2.71s/it]

Documents created: 99


In [ ]:
def chunk_text(text, chunk_size=900, overlap=150):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
chunks = []

for doc in documents:
    for page in doc["pages"]:
        page_chunks = chunk_text(page["text"])

        for chunk_number, chunk in enumerate(page_chunks, start=1):
            chunks.append({
                "chunk_id": len(chunks) + 1,
                "paper_id": doc["paper_id"],
                "title": doc["title"],
                "pdf_file": doc["pdf_file"],
                "page": page["page"],
                "chunk_number": chunk_number,
                "chunk_text": chunk
            })

chunks_df = pd.DataFrame(chunks)
chunks_df.head()

,chunk_id,paper_id,title,pdf_file,page,chunk_number,chunk_text
0,1,paper_001,,retrieval_augmented_generation_for_knowledge_i...,1,1,Retrieval-Augmented Generation for Knowledge-I...
1,2,paper_001,,retrieval_augmented_generation_for_knowledge_i...,2,1,The Divine Comedy (x) q Query Encoder q(x) MIP...
2,3,paper_001,,retrieval_augmented_generation_for_knowledge_i...,3,1,by θ that generates a current token based on a...
3,4,paper_001,,retrieval_augmented_generation_for_knowledge_i...,4,1,minimize the negative marginal log-likelihood ...
4,5,paper_001,,retrieval_augmented_generation_for_knowledge_i...,5,1,MSMARCO as an open-domain abstractive QA task....


In [ ]:
print("Total chunks:", len(chunks_df))
print("Total papers:", chunks_df["paper_id"].nunique())

chunks_df.head()

Total chunks: 2068
Total papers: 99


,chunk_id,paper_id,title,pdf_file,page,chunk_number,chunk_text
0,1,paper_001,,retrieval_augmented_generation_for_knowledge_i...,1,1,Retrieval-Augmented Generation for Knowledge-I...
1,2,paper_001,,retrieval_augmented_generation_for_knowledge_i...,2,1,The Divine Comedy (x) q Query Encoder q(x) MIP...
2,3,paper_001,,retrieval_augmented_generation_for_knowledge_i...,3,1,by θ that generates a current token based on a...
3,4,paper_001,,retrieval_augmented_generation_for_knowledge_i...,4,1,minimize the negative marginal log-likelihood ...
4,5,paper_001,,retrieval_augmented_generation_for_knowledge_i...,5,1,MSMARCO as an open-domain abstractive QA task....


In [ ]:
output_path = os.path.join(OUTPUT_DIR, "chunks.csv")

chunks_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /content/drive/MyDrive/CSAI415_D1/outputs/chunks.csv


In [ ]:
from google.colab import files

files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(chunks_df["paper_id"].nunique())

99


In [ ]:
chunks_df["chunk_text"].str.split().str.len().describe()

,chunk_text
count,2068.000000
mean,524.194874
std,245.867417
min,1.000000
25%,380.000000
50%,569.000000
75%,700.250000
max,900.000000


In [ ]:
chunks_df["word_count"] = chunks_df["chunk_text"].str.split().str.len()

chunks_df = chunks_df[chunks_df["word_count"] > 50]

print("Remaining chunks:", len(chunks_df))

Remaining chunks: 1958


In [ ]:
chunks_df = chunks_df.drop(columns=["word_count"])

In [ ]:
output_path = os.path.join(OUTPUT_DIR, "chunks_cleaned.csv")

chunks_df.to_csv(output_path, index=False)

print("Saved cleaned file!")

Saved cleaned file!


In [ ]:
from google.colab import files

files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

chunks_df = pd.read_csv("/content/drive/MyDrive/CSAI415_D1/outputs/chunks_cleaned.csv")

chunks_df.head()

,chunk_id,paper_id,title,pdf_file,page,chunk_number,chunk_text
0,1,paper_001,NaN,retrieval_augmented_generation_for_knowledge_i...,1,1,Retrieval-Augmented Generation for Knowledge-I...
1,2,paper_001,NaN,retrieval_augmented_generation_for_knowledge_i...,2,1,The Divine Comedy (x) q Query Encoder q(x) MIP...
2,3,paper_001,NaN,retrieval_augmented_generation_for_knowledge_i...,3,1,by θ that generates a current token based on a...
3,4,paper_001,NaN,retrieval_augmented_generation_for_knowledge_i...,4,1,minimize the negative marginal log-likelihood ...
4,5,paper_001,NaN,retrieval_augmented_generation_for_knowledge_i...,5,1,MSMARCO as an open-domain abstractive QA task....


In [ ]:
chunks_df["title"] = (
    chunks_df["pdf_file"]
    .astype(str)
    .str.replace(".pdf", "", regex=False)
    .str.replace("_", " ")
    .str.replace("-", " ")
)

chunks_df.head()

,chunk_id,paper_id,title,pdf_file,page,chunk_number,chunk_text
0,1,paper_001,retrieval augmented generation for knowledge i...,retrieval_augmented_generation_for_knowledge_i...,1,1,Retrieval-Augmented Generation for Knowledge-I...
1,2,paper_001,retrieval augmented generation for knowledge i...,retrieval_augmented_generation_for_knowledge_i...,2,1,The Divine Comedy (x) q Query Encoder q(x) MIP...
2,3,paper_001,retrieval augmented generation for knowledge i...,retrieval_augmented_generation_for_knowledge_i...,3,1,by θ that generates a current token based on a...
3,4,paper_001,retrieval augmented generation for knowledge i...,retrieval_augmented_generation_for_knowledge_i...,4,1,minimize the negative marginal log-likelihood ...
4,5,paper_001,retrieval augmented generation for knowledge i...,retrieval_augmented_generation_for_knowledge_i...,5,1,MSMARCO as an open-domain abstractive QA task....


In [ ]:
final_path = "/content/drive/MyDrive/CSAI415_D1/outputs/chunks_final.csv"

chunks_df.to_csv(final_path, index=False)

print("Saved final file!")

from google.colab import files

files.download(final_path)

Saved final file!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
chunks_df.sample(5)[["title", "page", "chunk_text"]]

,title,page,chunk_text
231,adaptive rag learning to adapt retrieval augme...,1,Adaptive-RAG: Learning to Adapt Retrieval-Augm...
1723,ares an automated evaluation framework for ret...,12,analyze the provided document and determine wh...
582,gtr large dual encoders are generalizable retr...,3,"poses a multi-vector encoding model, which rep..."
1183,think on graph deep and responsible reasoning ...,7,Published as a conference paper at ICLR 2024 Q...
1112,graphrag r1 graph retrieval augmented generati...,2,"WWW ’26, April 13–17, 2026, Dubai, United Arab..."


In [ ]:
print(chunks_df["paper_id"].nunique())

99
